In [7]:
import os
import logging
import aide

data_dir = r"c:\TUB\RDEP\workspaces\fairface_aide"
os.makedirs(data_dir, exist_ok=True)

GOAL = """
Build a machine learning pipeline using the FairFace dataset.

1. Load FairFace from HuggingFace (fields: image, race, gender, age).
2. Split into train / validation / test.
3. Train a model to predict race from the image.
4. Evaluate fairness by race group on the test set (Accuracy, Precision, Recall, FPR, FNR).
5. Save metrics and plots.
"""

EVAL = """
Macro F1 score on the test split for race classification.
"""

def main():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )

    exp = aide.Experiment(
        data_dir=data_dir,
        goal=GOAL,
        eval=EVAL,
    )

    best = exp.run(steps=3)

    print("\n=== Raw solution object ===")
    print(best)

    print("\n=== Public attributes on solution ===")
    names = [n for n in dir(best) if not n.startswith("_")]
    for name in sorted(names):
        try:
            value = getattr(best, name)
        except Exception as e:
            value = f"<error reading attribute: {e}>"
        print(f"{name}: {value}")

    if hasattr(best, "code"):
        print("\n=== Generated code ===\n")
        print(best.code)

if __name__ == "__main__":
    main()



=== Raw solution object ===
Solution(code='import os\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision import transforms\nimport timm\nfrom datasets import load_dataset\nfrom sklearn.metrics import f1_score\nfrom PIL import Image\n\n# 1. Load and split dataset\nhf = load_dataset("fairface", cache_dir="./input")["train"]\nsplit1 = hf.train_test_split(test_size=0.2, seed=42)\ntrain_hf = split1["train"]\ntemp = split1["test"]\nsplit2 = temp.train_test_split(test_size=0.5, seed=42)\nval_hf, test_hf = split2["train"], split2["test"]\n\n# Label mapping\nclasses = train_hf.features["race"].names\nlabel2id = {c: i for i, c in enumerate(classes)}\nid2label = {i: c for c, i in label2id.items()}\n\n\n# 2. Dataset class\nclass FaceDataset(Dataset):\n    def __init__(self, hf_ds, transform):\n        self.ds = hf_ds\n        self.transform = transform\n\n    def __len__(self):\n        return l

In [17]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from datasets import load_dataset, Image as HFImage
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from PIL import Image


# ---------------------------
# 1. Load and subset FairFace
# ---------------------------
config_name = "0.25"  # or "1.25"
ds = load_dataset("HuggingFaceM4/FairFace", config_name, cache_dir="./input")
hf = ds["train"]

# Auto-detect the image column (first column with HF Image feature)
image_column = None
for name, feat in hf.features.items():
    if isinstance(feat, HFImage):
        image_column = name
        break

if image_column is None:
    raise RuntimeError("Could not find an Image column in the dataset features.")

print("Detected image column:", image_column)

# Race label column
label_column = "race"
label_feature = hf.features[label_column]

# If class names exist, use them; otherwise build from unique values
if getattr(label_feature, "names", None) is not None:
    classes = label_feature.names
else:
    unique_vals = sorted(set(hf[label_column]))
    classes = [str(v) for v in unique_vals]

label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}

print("Classes:", classes)

# Train/val/test split on full dataset
split1 = hf.train_test_split(test_size=0.2, seed=42)
train_hf = split1["train"]
temp = split1["test"]
split2 = temp.train_test_split(test_size=0.5, seed=42)
val_hf, test_hf = split2["train"], split2["test"]

# ---------------------------
# Limit to about 2000 samples
# ---------------------------
# You can change these numbers if you want more/less data.
MAX_TRAIN = 1200
MAX_VAL = 400
MAX_TEST = 400  # total ≈ 2000

if len(train_hf) > MAX_TRAIN:
    train_hf = train_hf.select(range(MAX_TRAIN))
if len(val_hf) > MAX_VAL:
    val_hf = val_hf.select(range(MAX_VAL))
if len(test_hf) > MAX_TEST:
    test_hf = test_hf.select(range(MAX_TEST))

print("Final subset sizes ->",
      "train:", len(train_hf),
      "val:", len(val_hf),
      "test:", len(test_hf))


# ---------------------------
# 2. PyTorch Dataset wrapper
# ---------------------------
class FaceDataset(Dataset):
    """Convert FairFace HuggingFace dataset into a PyTorch Dataset."""

    def __init__(self, hf_ds, transform, image_col, label_col):
        self.ds = hf_ds
        self.transform = transform
        self.image_col = image_col
        self.label_col = label_col

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        # Image
        img = item[self.image_col]
        if isinstance(img, Image.Image):
            pil = img
        else:
            pil = Image.fromarray(img)

        x = self.transform(pil)

        # Label can be a string ("White") or an integer id (0,1,...)
        raw_label = item[self.label_col]
        if isinstance(raw_label, str):
            y = label2id[raw_label]
        else:
            y = int(raw_label)

        path = getattr(img, "path", "")
        return x, y, path


# ---------------------------
# 3. Transforms & Dataloaders
# ---------------------------
train_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

eval_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

train_ds = FaceDataset(train_hf, train_tf, image_column, label_column)
val_ds = FaceDataset(val_hf, eval_tf, image_column, label_column)
test_ds = FaceDataset(test_hf, eval_tf, image_column, label_column)

# num_workers=0 is safer on Windows / notebook
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)


# ---------------------------
# 4. Model, loss, optimizer
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model("resnet18", pretrained=True, num_classes=len(classes)).to(
    device
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


# ---------------------------
# 5. Training loop (1 epoch)
# ---------------------------
epochs = 1  # quick run: only 1 epoch

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x, y, _ in train_dl:
        x, y = x.to(device), y.to(device)
        preds = model(x)
        loss = criterion(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    avg_loss = running_loss / len(train_dl.dataset)

    # Validation macro F1
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y, _ in val_dl:
            x = x.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1).cpu().numpy()
            val_preds.extend(pred)
            val_labels.extend(y.numpy())

    val_f1 = f1_score(val_labels, val_preds, average="macro")
    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train loss: {avg_loss:.4f} | Val Macro F1: {val_f1:.4f}"
    )


# ---------------------------
# 6. Test predictions
# ---------------------------
model.eval()
test_preds, test_labels, test_paths = [], [], []

with torch.no_grad():
    for x, y, paths in test_dl:
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        test_preds.extend(pred)
        test_labels.extend(y.numpy())
        test_paths.extend(paths)

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)


# ---------------------------
# 7. Overall performance
# ---------------------------
overall_acc = (test_preds == test_labels).mean()
overall_f1 = f1_score(test_labels, test_preds, average="macro")

print("\n=== Overall Test Performance (subset) ===")
print(f"Accuracy: {overall_acc:.4f}")
print(f"Macro F1: {overall_f1:.4f}")


# ---------------------------
# 8. Fairness metrics per race
# ---------------------------
cm = confusion_matrix(test_labels, test_preds, labels=list(range(len(classes))))
precision, recall, f1_vals, support = precision_recall_fscore_support(
    test_labels, test_preds, labels=list(range(len(classes))), zero_division=0
)

FPR, FNR = [], []
for i in range(len(classes)):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    FPR.append(fpr)
    FNR.append(fnr)

fairness_df = pd.DataFrame(
    {
        "race": classes,
        "support": support,
        "precision": precision,
        "recall": recall,
        "f1": f1_vals,
        "FPR": FPR,
        "FNR": FNR,
    }
)

print("\n=== Fairness Metrics by Race (subset) ===")
print(fairness_df)


# ---------------------------
# 9. Save outputs
# ---------------------------
os.makedirs("working", exist_ok=True)

fairness_df.to_csv("working/fairface_fairness_metrics_2k.csv", index=False)

df_pred = pd.DataFrame(
    {
        "image_path": test_paths,
        "true_race": [id2label[int(t)] for t in test_labels],
        "predicted_race": [id2label[int(p)] for p in test_preds],
    }
)
df_pred.to_csv("working/predictions_2k.csv", index=False)

print("\nSaved fairness metrics and predictions (subset) to 'working' folder.")


Detected image column: image
Classes: ['East Asian', 'Indian', 'Black', 'White', 'Middle Eastern', 'Latino_Hispanic', 'Southeast Asian']
Final subset sizes -> train: 1200 val: 400 test: 400
Epoch 1/1 | Train loss: 1.9386 | Val Macro F1: 0.1303

=== Overall Test Performance (subset) ===
Accuracy: 0.1775
Macro F1: 0.1293

=== Fairness Metrics by Race (subset) ===
              race  support  precision    recall        f1       FPR       FNR
0       East Asian       54   0.119403  0.148148  0.132231  0.170520  0.851852
1           Indian       58   0.461538  0.103448  0.169014  0.020468  0.896552
2            Black       53   0.041667  0.018868  0.025974  0.066282  0.981132
3            White       78   0.217687  0.410256  0.284444  0.357143  0.589744
4   Middle Eastern       40   0.200000  0.050000  0.080000  0.022222  0.950000
5  Latino_Hispanic       67   0.158273  0.328358  0.213592  0.351351  0.671642
6  Southeast Asian       50   0.000000  0.000000  0.000000  0.000000  1.000000

Sav

In [22]:
"""
Use AIDE ML to automatically generate a fair face classification model
based on the FairFace dataset. The goal is not only accuracy but also
reducing racial bias using fairness-aware loss terms.

All comments must be in English.
"""

import logging
from pathlib import Path

import aide


# -------------------------------------------------------------------
#  FAIRNESS-AWARE AIDE GENERATION PROMPT
# -------------------------------------------------------------------
FAIRFACE_GOAL = """
You are an expert ML fairness researcher. Generate a Python script (single cell)
that trains a face classification model on a 2k subset of the FairFace dataset
AND optimizes for racial fairness, not only accuracy.

Your model MUST use the following fairness formulas from the literature:

=====================================================
(1) Equal Opportunity Regularizer (Hardt et al.)
=====================================================
For each group g in {1,...,G}:

    TPR_g = TP_g / (TP_g + FN_g)

Average TPR:

    TPR_bar = (1/G) * sum_g TPR_g

Fairness penalty:

    L_EO = sum_g (TPR_g - TPR_bar)^2

=====================================================
(2) Equalized Odds (Hardt et al.)
=====================================================
For each group g:

    FPR_g = FP_g / (FP_g + TN_g)

Average FPR:

    FPR_bar = (1/G) * sum_g FPR_g

Penalty:

    L_EOdds = sum_g [ (TPR_g - TPR_bar)^2 + (FPR_g - FPR_bar)^2 ]

=====================================================
(3) Group DRO (Sagawa et al.)
=====================================================
Let L_g be the loss for group g, or equivalently 1 - F1_g.

Group DRO objective:

    L_DRO = max_g L_g

Equivalently, the goal is to maximize the worst-group F1:

    maximize min_g F1_g

=====================================================
(4) Class-Balanced Loss (Cui et al.)
=====================================================
For group g with n_g samples:

    E_g = (1 - beta^(n_g)) / (1 - beta)

Class weight:

    w_g = 1 / E_g

Then:

    L_CB = w_y * CE(p, y)

=====================================================
(5) Adversarial Debiasing (Zhang et al.)
=====================================================
Main encoder produces features h. An adversarial head tries to
predict the sensitive group g:

    L_adv = CE(D(h), g)

Total loss:

    L = L_task - lambda_adv * L_adv

=====================================================
(6) Adaptive Margin Adjustment
=====================================================
For each group g, given its current F1_g:

    m_g = m0 + alpha * (1 - F1_g)

This larger margin can be applied to underperforming groups, e.g.
in a margin-based softmax (ArcFace-style) head.

=====================================================
(7) FINAL OPTIMIZATION OBJECTIVE
=====================================================
The model should optimize the total loss:

    L_total =
        L_CE
      + lambda1 * L_EO
      + lambda2 * L_EOdds
      + lambda3 * L_CB
      + lambda4 * L_adv
      + lambda5 * L_DRO

=====================================================
DATA AND TRAINING REQUIREMENTS
=====================================================

1. Dataset:
   - Use: load_dataset("HuggingFaceM4/FairFace", "0.25", cache_dir="./input")
   - Use the "train" split and then create train/validation/test splits
     with a fixed random seed (e.g. 42).

2. Subset size:
   - After splitting, limit to at most:
       * 1200 samples for training
       * 400 samples for validation
       * 400 samples for test
   - Print: "Final subset sizes -> train: ... val: ... test: ..."

3. Columns:
   - Image column: "image"
   - Label column: "race"
   - Race classes (7):
       ["East Asian", "Indian", "Black", "White",
        "Middle Eastern", "Latino_Hispanic", "Southeast Asian"]
   - Build:
       label2id: dict from class name to integer id
       id2label: inverse mapping

4. PyTorch Dataset and DataLoaders:
   - Define FaceDataset(Dataset) that:
       * receives a HF dataset split, transform, image_col, label_col
       * __getitem__ returns (image_tensor, label_id, image_path)
       * converts numpy → PIL with Image.fromarray if needed
       * converts labels to int ids (using label2id or direct int)
   - Transforms:
       * Resize(224,224), RandomHorizontalFlip (train only),
         ToTensor, Normalize(ImageNet mean/std)
   - DataLoaders:
       * batch_size = 32
       * shuffle = True for train, False for val/test
       * num_workers = 0 (important on Windows)
       * apply class-balanced sampling or oversampling for fairness.

5. Model:
   - Use a lightweight backbone (e.g. ResNet18 from timm)
   - The classifier head must have num_classes = 7.
   - Optionally add:
       * an adversarial head to predict the race group
       * an adaptive margin mechanism for weak groups.

6. Training:
   - Train exactly 1 epoch (for quick experiments).
   - Use Adam optimizer (lr = 1e-4) or similar.
   - During training, compute the main CE loss and, where feasible,
     add fairness terms (L_EO, L_EOdds, L_CB, L_adv, L_DRO) into L_total.
   - At the end of the epoch:
       * print the average training loss
       * compute validation macro F1.

7. Test-time fairness evaluation:
   - On the test set, collect:
       * test_preds (integer class ids)
       * test_labels
       * test_paths (image paths if available)
   - Compute for each race group:
       * support (sample count)
       * precision, recall, F1
       * FPR, FNR:
           FPR_g = FP_g / (FP_g + TN_g) if > 0 else 0
           FNR_g = FN_g / (FN_g + TP_g) if > 0 else 0
   - Also compute:
       * overall accuracy
       * overall macro F1
       * worst-group F1 (min_g F1_g)
       * TPR/FPR gaps across groups.

8. Output and saving:
   - Print:
       * overall accuracy and macro F1
       * worst-group F1
       * fairness penalty values (e.g. L_EO, L_EOdds, L_DRO)
       * a pandas DataFrame:
           ["race", "support", "precision", "recall", "f1", "FPR", "FNR"]
   - Create a folder "working" if it does not exist.
   - Save:
       * fairness metrics to "working/fairface_fairness_metrics_2k.csv"
       * per-image predictions to "working/predictions_2k.csv"

9. Implementation constraints:
   - All comments must be in English.
   - Code must run end-to-end in a Jupyter notebook on Windows.
   - Use only standard Python + torch, torchvision or timm, datasets,
     numpy, pandas, sklearn, PIL.
   - Keep everything in a single code cell (no external file dependencies).
   - Do not use multiprocessing (num_workers = 0).
   - Return only the final Python code as a single string.
"""


# -------------------------------------------------------------------
#  AIDE EVALUATION PROMPT
# -------------------------------------------------------------------
FAIRFACE_EVAL = """
You evaluate candidate solutions for training a fair face classifier
on a 2k subset of FairFace.

Score a solution higher when:
1. It correctly loads the dataset with:
   load_dataset("HuggingFaceM4/FairFace", "0.25", cache_dir="./input")
   and uses the "train" split with a fixed random seed for splitting.
2. It creates train/validation/test with subset sizes around:
   1200 train, 400 validation, 400 test (or fewer if necessary).
3. It uses the correct columns: "image" and "race", and the 7 race classes.
4. It defines a proper FaceDataset and DataLoaders with:
   batch_size = 32, num_workers = 0, and some form of class-balanced
   sampling or weighting to improve fairness.
5. It implements fairness-aware training:
   - uses or approximates the formulas for L_EO, L_EOdds, L_CB, L_adv, L_DRO
   - or at least clearly aims to reduce TPR/FPR gaps and improve
     worst-group F1, not only overall accuracy.
6. It computes and prints:
   - overall accuracy and macro F1 on the test set
   - worst-group F1
   - per-race support, precision, recall, F1, FPR, FNR.
7. It saves the required CSV files in the "working" folder.
8. The code is clear, commented in English, and likely to run on a
   Windows Jupyter notebook environment.

Penalize solutions that:
- Ignore group fairness and only maximize accuracy.
- Fail to compute per-race performance metrics.
- Use the wrong dataset or columns.
- Use multiprocessing workers in DataLoaders.
- Do not save the CSV outputs as requested.
"""


def main():
    """Run the AIDE experiment and print the best generated code."""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    )

    # Use a workspace inside the current project directory
    data_dir = Path("workspaces/fairface_aide_fair").resolve()
    print("Using data_dir:", data_dir)

    # *** IMPORTANT: make sure the directory exists ***
    data_dir.mkdir(parents=True, exist_ok=True)

    exp = aide.Experiment(
        data_dir=str(data_dir),
        goal=FAIRFACE_GOAL,
        eval=FAIRFACE_EVAL,
    )

    # Number of improvement steps; you can increase this later
    best = exp.run(steps=3)

    print("\n=== Best solution object ===")
    print(best)

    if hasattr(best, "code"):
        print("\n=== Best solution code ===\n")
        print(best.code)

        # Optionally save generated code
        out_path = data_dir / "generated_fairface_fair_model.py"
        out_path.write_text(best.code, encoding="utf-8")
        print(f"\nSaved generated code to: {out_path}")


if __name__ == "__main__":
    main()

Using data_dir: C:\TUB\RDEP\workspaces\fairface_aide_fair


C:\Users\wenyi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\aide\utils\tree_export.py:34: RuntimeWarning: invalid value encountered in divide
  layout = (layout - layout.min(axis=0)) / (layout.max(axis=0) - layout.min(axis=0))



=== Best solution object ===
Solution(code='import os\nimport torch\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom datasets import load_dataset\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\nimport torchvision.transforms as T\nimport timm\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support\n\n# 1. Load and split the dataset\nds_full = load_dataset("HuggingFaceM4/FairFace", "0.25", cache_dir="./input")["train"]\nds2k = ds_full.shuffle(seed=42).select(range(2000))\nsplit1 = ds2k.train_test_split(test_size=800, seed=42)\ntrain_ds = split1["train"]\ntmp_ds = split1["test"]\nsplit2 = tmp_ds.train_test_split(test_size=400, seed=42)\nval_ds = split2["train"]\ntest_ds = split2["test"]\nprint(\n    f"Final subset sizes -> train: {len(train_ds)} val: {len(val_ds)} test: {len(test_ds)}"\n)\n\n# 2. Label mapping\nclasses = [\n    "East Asian",\n    "Indian",\n    "Black",\n    "White",\n    "Middle Eastern",\n    "Latino_H

In [11]:
! pip install "huggingface_hub[hf_xet]"


  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl (2.9 MB)


In [24]:
# ===============================================================
# FairFace Fairness Evaluation for NEW MODEL (no training here)
# This cell assumes that the following variables ALREADY exist:
#   - test_labels : array-like of shape (n_samples,)
#   - test_preds  : array-like of shape (n_samples,)
#   - test_paths  : list of str (can be "")
#
# It will:
#   - compute overall accuracy & macro F1
#   - compute per-race precision, recall, F1, FPR, FNR
#   - save results to "working/fairface_fairness_metrics_fair_model.csv"
#     and "working/predictions_fair_model.csv"
# ===============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

# 1. Define FairFace race classes
classes = [
    "East Asian",
    "Indian",
    "Black",
    "White",
    "Middle Eastern",
    "Latino_Hispanic",
    "Southeast Asian",
]
id2label = {i: c for i, c in enumerate(classes)}


# 2. Fairness evaluation function
def evaluate_fairness_and_save(
    test_labels,
    test_preds,
    test_paths,
    classes,
    id2label,
    save_dir="working",
    suffix="fair_model",
):
    """
    Evaluate racial fairness for the FairFace task and save results.
    """

    os.makedirs(save_dir, exist_ok=True)

    # Convert to numpy arrays
    test_labels = np.asarray(test_labels)
    test_preds = np.asarray(test_preds)

    # Safety check
    if test_labels.shape[0] != test_preds.shape[0]:
        raise ValueError(
            f"Length mismatch: test_labels={test_labels.shape[0]}, "
            f"test_preds={test_preds.shape[0]}"
        )

    # -------------------------
    # Overall metrics
    # -------------------------
    overall_acc = (test_preds == test_labels).mean()
    overall_f1 = f1_score(test_labels, test_preds, average="macro")

    print(f"\n=== Overall Test Performance ({suffix}) ===")
    print(f"Accuracy: {overall_acc:.4f}")
    print(f"Macro F1: {overall_f1:.4f}")

    # -------------------------
    # Per-class fairness metrics
    # -------------------------
    cm = confusion_matrix(test_labels, test_preds, labels=list(range(len(classes))))
    precision, recall, f1_vals, support = precision_recall_fscore_support(
        test_labels,
        test_preds,
        labels=list(range(len(classes))),
        zero_division=0,
    )

    FPR, FNR = [], []
    for i in range(len(classes)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        FPR.append(fpr)
        FNR.append(fnr)

    fairness_df = pd.DataFrame(
        {
            "race": classes,
            "support": support,
            "precision": precision,
            "recall": recall,
            "f1": f1_vals,
            "FPR": FPR,
            "FNR": FNR,
        }
    )

    print(f"\n=== Fairness Metrics by Race ({suffix}) ===")
    print(fairness_df)

    # -------------------------
    # Save CSVs
    # -------------------------
    metrics_path = os.path.join(save_dir, f"fairface_fairness_metrics_{suffix}.csv")
    preds_path = os.path.join(save_dir, f"predictions_{suffix}.csv")

    fairness_df.to_csv(metrics_path, index=False)

    # If test_paths length mismatches, pad or cut
    if len(test_paths) != len(test_labels):
        print(
            f"[Warning] len(test_paths)={len(test_paths)} != len(test_labels)={len(test_labels)}."
            " Paths will be truncated or padded with empty strings."
        )
        paths_fixed = list(test_paths)[: len(test_labels)]
        if len(paths_fixed) < len(test_labels):
            paths_fixed += [""] * (len(test_labels) - len(paths_fixed))
    else:
        paths_fixed = test_paths

    df_pred = pd.DataFrame(
        {
            "image_path": paths_fixed,
            "true_race": [id2label[int(t)] for t in test_labels],
            "predicted_race": [id2label[int(p)] for p in test_preds],
        }
    )
    df_pred.to_csv(preds_path, index=False)

    print(f"\nSaved fairness metrics to: {metrics_path}")
    print(f"Saved predictions to:     {preds_path}")

    return fairness_df


# 3. Run fairness evaluation on your NEW MODEL outputs
#    (here we assume test_labels / test_preds / test_paths
#     already exist from your model script)

fairness_results = evaluate_fairness_and_save(
    test_labels=test_labels,   # <-- from your new model
    test_preds=test_preds,     # <-- from your new model
    test_paths=test_paths,     # <-- from your new model
    classes=classes,
    id2label=id2label,
    save_dir="working",
    suffix="fair_model",       # name tag for this model
)


=== Overall Test Performance (fair_model) ===
Accuracy: 0.1725
Macro F1: 0.1226

=== Fairness Metrics by Race (fair_model) ===
              race  support  precision    recall        f1       FPR       FNR
0       East Asian       54   0.146552  0.314815  0.200000  0.286127  0.685185
1           Indian       58   0.181818  0.068966  0.100000  0.052632  0.931034
2            Black       53   0.111111  0.018868  0.032258  0.023055  0.981132
3            White       78   0.220779  0.435897  0.293103  0.372671  0.564103
4   Middle Eastern       40   0.000000  0.000000  0.000000  0.000000  1.000000
5  Latino_Hispanic       67   0.148148  0.059701  0.085106  0.069069  0.940299
6  Southeast Asian       50   0.125000  0.180000  0.147541  0.180000  0.820000

Saved fairness metrics to: working\fairface_fairness_metrics_fair_model.csv
Saved predictions to:     working\predictions_fair_model.csv
